# Semiconductor GraphRAG — Exploration Notebook

Use this notebook to:
- Verify Neo4j schema and node counts
- Test raw Cypher queries
- Run vector similarity searches
- Invoke the full LangGraph agent

**Prerequisites:** Docker Neo4j running + data ingested via `scripts/ingest.py`

In [ ]:
import sys
sys.path.insert(0, '..')

from graphrag.graph_db import get_driver, run_query, get_neo4j_graph
from graphrag.config import NEO4J_URI, GEMINI_LLM_MODEL

## 1. Verify Node Counts

In [ ]:
driver = get_driver()

for label in ['Input', 'Provider', 'Stage', 'SpecChunk']:
    result = run_query(driver, f'MATCH (n:{label}) RETURN count(n) AS cnt')
    print(f"{label:12s}: {result[0]['cnt']}")

for rel in ['PROVIDES', 'GOES_INTO', 'IS_TYPE_OF', 'IN_STAGE', 'DESCRIBES']:
    result = run_query(driver, f'MATCH ()-[r:{rel}]->() RETURN count(r) AS cnt')
    print(f"{rel:12s}: {result[0]['cnt']}")

driver.close()

## 2. Neo4j Schema (for prompt verification)

In [ ]:
graph = get_neo4j_graph()
print(graph.schema)

## 3. Raw Cypher Queries

In [ ]:
# ASML monopoly check
driver = get_driver()
results = run_query(driver, """
    MATCH (p:Provider)-[r:PROVIDES]->(i:Input)
    WHERE toLower(i.input_name) CONTAINS 'euv'
    RETURN p.provider_name, p.country, p.provider_type, r.share_provided, r.year
    ORDER BY r.share_provided DESC LIMIT 10
""")
driver.close()
for r in results:
    print(r)

In [ ]:
# Lithography taxonomy
driver = get_driver()
results = run_query(driver, """
    MATCH (sub:Input)-[:IS_TYPE_OF*1..3]->(parent:Input)
    WHERE toLower(parent.input_name) CONTAINS 'lithography'
    RETURN sub.input_name AS subtype, sub.type, parent.input_name AS parent
    ORDER BY sub.input_name LIMIT 20
""")
driver.close()
for r in results:
    print(r)

In [ ]:
# Single-country monopolies
driver = get_driver()
results = run_query(driver, """
    MATCH (p:Provider {provider_type: 'country'})-[r:PROVIDES]->(i:Input)
    WHERE r.share_provided >= 100
    RETURN i.input_name AS input, p.provider_name AS country, r.share_provided AS share
    ORDER BY i.input_name LIMIT 20
""")
driver.close()
for r in results:
    print(r)

## 4. Text-to-Cypher Chain

In [ ]:
from graphrag.retrieval.graph_retriever import get_graph_retriever, graph_query

chain = get_graph_retriever()
result = graph_query(chain, 'What are all the subtypes of Lithography tools?')
print('Generated Cypher:', result['cypher'])
print('\nRaw results:')
for r in result['raw_results']:
    print(r)

## 5. Vector Search

In [ ]:
from graphrag.retrieval.vector_retriever import get_vector_store, vector_search

store = get_vector_store()
results = vector_search(store, 'photomask defect inspection', k=5)
for r in results:
    print(r)
    print('---')

## 6. Full Agent

In [ ]:
import asyncio
from graphrag.agents.graph import ask

questions = [
    'Who supplies EUV lithography tools and what is their market share?',
    'What are all the subtypes of Lithography tools?',
    'Which inputs are sourced entirely from one country?',
    'Show me the full production pipeline from Chip design to Finished logic chip.',
]

for q in questions:
    print(f'\nQ: {q}')
    print('-' * 60)
    answer = asyncio.run(ask(q))
    print(answer)